# Week 3 – Topic 3 Lab: Parts A & B (PDF/TXT pathway, real data: CUAD)
**Project:** AI-Powered Contract and Terms & Conditions Analyzer

**Data:** CUAD v1 (Contract Understanding Atticus Dataset), 510 real commercial contracts, licence **CC BY 4.0**. Hendrycks, Burns, Chen & Ball (2021), *CUAD: An Expert-Annotated NLP Dataset for Legal Contract Review*, arXiv:2103.06268. Source: https://huggingface.co/datasets/theatticusproject/cuad

Run the cells **in order**. This notebook prepares text only. It does not measure clause-detection accuracy.

| Cell | Lab step |
|---|---|
| 1, 2, 3 | A2 – workspace, download CUAD files, verify |
| 4, 4A | B2 – source selection + lineage helpers |
| A3 | A3 – inspect two files |
| 5A1, 5A2 | B2/B3 – Pathway A: clean, compare chunking, export |
| B4.1, 6A, 6B | B4 – lineage registration, manifest, verification |

## A2 – Workspace setup

In [ ]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

In [ ]:
# Cell 2 — Download two CUAD contracts (one PDF, one TXT) from Hugging Face
!pip -q install pypdf huggingface_hub
from huggingface_hub import HfApi, hf_hub_download
from pypdf import PdfReader

REPO = "theatticusproject/cuad"
all_files = HfApi().list_repo_files(REPO, repo_type="dataset")
pdfs = sorted(f for f in all_files if f.startswith("CUAD_v1/full_contract_pdf/") and f.lower().endswith(".pdf"))
txts = sorted(f for f in all_files if f.startswith("CUAD_v1/full_contract_txt/") and f.lower().endswith(".txt"))
print("Contracts available -> PDF:", len(pdfs), "| TXT:", len(txts))

# PDF: take the first PDF that has selectable text (some CUAD PDFs are scanned images and would need OCR)
PDF_REL = None
for name in pdfs[:10]:
    local = Path(hf_hub_download(REPO, name, repo_type="dataset", local_dir=RAW_DIR))
    n_chars = sum(len(pg.extract_text() or "") for pg in PdfReader(str(local)).pages)
    print(f"  tried {local.name}: {n_chars} extracted characters")
    if n_chars >= 1000:
        PDF_REL = str(local.relative_to(RAW_DIR)); break
    local.unlink()  # scanned / empty: remove and try the next one
assert PDF_REL, "No text-based PDF found in the first 10 - increase the range or add OCR"

# TXT: a different contract from the middle of the list (deterministic, so reruns match)
txt_local = Path(hf_hub_download(REPO, txts[len(txts) // 2], repo_type="dataset", local_dir=RAW_DIR))
TXT_REL = str(txt_local.relative_to(RAW_DIR))
print("Selected PDF:", PDF_REL)
print("Selected TXT:", TXT_REL)

In [ ]:
# Cell 3 — Confirm downloaded inputs
files = sorted(p for p in RAW_DIR.rglob("*") if p.is_file())
print("Input count:", len(files))
for p in files:
    print(f"- {p.relative_to(RAW_DIR)}  ({p.stat().st_size} bytes)")
assert files, "No files in RAW_DIR"
print("Upload verified. Select your relevant input(s) for Step A3.")

## B2 – Common helpers (Cell 4 and 4A)

In [ ]:
# Cell 4 — Source selection + lineage utilities
from pathlib import Path
import json, hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")
print("Available functions: select_source(), source_hash(), save_manifest()")

In [ ]:
# Cell 4A — Common lineage registry
lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

## A1 – Requirement → data mapping (Topic 2)

| Requirement ID | Needed information / feature | Candidate source | Why usable / constraint |
|---|---|---|---|
| **FR-03 / FR-05** (text extraction; cleaning and segmentation with context preserved) | Readable text from real contract PDF/TXT, split into clause-sized segments that keep their headings | **CUAD v1** full-contract PDFs and TXTs (real commercial contracts) | Real contract wording and layout. Limitation: commercial agreements, not consumer T&Cs; some PDFs are scanned images (OCR out of V1 scope). |
| **NFR-04 / FR-06–07** (accuracy evaluated on a labelled test set; clause identification and categorisation) | Expert clause labels for later evaluation | CUAD `master_clauses.csv` / `CUAD_v1.json` (41 clause categories) | Labels are not used in Topic 3; they are why CUAD suits later evaluation. |
| **NFR-06 / RAI-04** (public or authorised test documents) and **RAI-01 / NFR-05** (source grounding) | Licence, source identity, checksum, chunk-to-source link | CUAD (CC BY 4.0, attribution required) + manifest | Redistributable with attribution. Contracts originate from public filings; no private documents are used. |

## A3 – Select and inspect two sources

In [ ]:
# Cell A3 — Inspect two representative sources (PDF_REL / TXT_REL were chosen in Cell 2)
import re

pdf_path = select_source(relative_path=PDF_REL)
txt_path = select_source(relative_path=TXT_REL)
SOURCES = [pdf_path, txt_path]

def read_pages(path):
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
    if path.suffix.lower() == ".txt":
        return [(1, path.read_text(encoding="utf-8", errors="strict"))]
    raise ValueError(f"Pathway A needs PDF or TXT, got {path.suffix}")

for p in SOURCES:
    pages = read_pages(p)
    text = "\n".join(t for _, t in pages)
    print("=" * 60)
    print("Source:", p.relative_to(RAW_DIR))
    print("Type:", p.suffix.lower(), "| bytes:", p.stat().st_size)
    print("Pages/segments:", len(pages), "| extracted characters:", len(text))
    print("SHA-256:", source_hash(p))
    print("Content sample:", repr(text[:400]))
    assert text.strip(), "No usable text: this file may be scanned and need OCR."

## Pathway A – clean and compare two chunking approaches

In [ ]:
# Cell 5A1 — Clean + compare fixed-size vs structure-aware chunking
import statistics

FIXED_SIZE = 1000      # baseline: cut every 1000 characters
MAX_CHUNK = 3000       # structure-aware chunks longer than this are split at blank lines
MIN_CHUNK = 80         # tiny pieces (e.g. contents-page lines) are merged into the next chunk

def clean_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x0c", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def fixed_chunks(s, size=FIXED_SIZE):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size].strip() for i in range(0, len(s), size) if s[i:i + size].strip()]

# Clause / section headings: "ARTICLE IV", "Section 2.1", "1. Term", "2.3 Fees"
HEADING = re.compile(
    r"^\s*(ARTICLE\s+[IVXLC\d]+\b.*"
    r"|SECTION\s+\d+(\.\d+)*\b.*|Section\s+\d+(\.\d+)*\b.*"
    r"|\d{1,2}(\.\d{1,2})*\.?\s+[A-Z][A-Za-z ,&/\-]{2,60}\.?\s*)$"
)

def split_long(chunk, limit=MAX_CHUNK):
    if len(chunk) <= limit:
        return [chunk]
    out, cur = [], ""
    for para in re.split(r"\n\s*\n", chunk):
        if cur and len(cur) + len(para) + 2 > limit:
            out.append(cur.strip()); cur = para
        else:
            cur = (cur + "\n\n" + para) if cur else para
    if cur.strip():
        out.append(cur.strip())
    # any single paragraph still too long falls back to fixed cuts
    final = []
    for c in out:
        final.extend([c] if len(c) <= limit else fixed_chunks(c, limit))
    return final

def structure_chunks(s):
    chunks, current = [], []
    for line in s.splitlines():
        if HEADING.match(line) and any(l.strip() for l in current):
            chunks.append("\n".join(current).strip()); current = [line]
        else:
            current.append(line)
    if any(l.strip() for l in current):
        chunks.append("\n".join(current).strip())
    if len(chunks) < 3:   # no usable headings found: fall back to paragraph blocks
        chunks = [b.strip() for b in re.split(r"\n\s*\n", s) if b.strip()]
    merged, carry = [], ""
    for c in chunks:
        c = (carry + "\n" + c).strip() if carry else c
        if len(c) < MIN_CHUNK:
            carry = c
        else:
            merged.append(c); carry = ""
    if carry:
        merged.append(carry) if not merged else merged.__setitem__(-1, merged[-1] + "\n" + carry)
    out = []
    for c in merged:
        out.extend(split_long(c))
    return out

prepared = {}
for p in SOURCES:
    text = clean_text("\n".join(t for _, t in read_pages(p)))
    fx, st = fixed_chunks(text), structure_chunks(text)
    prepared[p.name] = {"clean": text, "fixed": fx, "structure": st}
    print("=" * 60, "\n", p.name, "| cleaned chars:", len(text))
    for label, ch in (("Fixed", fx), ("Structure-aware", st)):
        sizes = [len(c) for c in ch]
        print(f"{label:16} chunks={len(ch):4}  min={min(sizes)}  median={int(statistics.median(sizes))}  max={max(sizes)}")
    print("-- structure sample:")
    for c in st[:3]:
        print("   ", repr(c[:110]))

In [ ]:
# B3 comparison table (observed evidence)
print(f"{'Source':45} {'Fixed':>6} {'Struct':>7}  Fixed chunks starting mid-sentence")
for name, d in prepared.items():
    mid = sum(1 for c in d["fixed"] if c[:1].islower())
    print(f"{name[:44]:45} {len(d['fixed']):>6} {len(d['structure']):>7}  {mid}/{len(d['fixed'])}")
print()
print("Look at 2-3 structure-aware chunks yourself: does each start at a clause heading?")
for name, d in prepared.items():
    starts = sum(1 for c in d["structure"] if HEADING.match(c.splitlines()[0]))
    print(f"{name[:44]:45} structure chunks starting at a heading: {starts}/{len(d['structure'])}")

**B3 decision (read the numbers above, then edit this in your own words):**
Selected approach: **structure-aware chunking** (split at clause/section headings, merge tiny fragments, cap long chunks at blank lines). *Benefit:* each chunk keeps a clause together with its heading, which supports FR-05 (preserve context) and RAI-01 / NFR-05 (a finding can be traced to a clause). Fixed 1000-character chunks cut clauses mid-sentence (see the mid-sentence count). *Accepted limitation:* heading detection is rule-based, so contracts with unusual numbering may be split imperfectly; the fallback is blank-line paragraphs. Only two contracts were inspected, so this is a qualitative choice, not a measured accuracy result.

In [ ]:
# Cell 5A2 — Export prepared records (selected approach) + B3 validation
approach = (f"structure-aware chunking: split at clause/section headings (ARTICLE/Section/numbered), "
            f"merge fragments < {MIN_CHUNK} chars, cap at {MAX_CHUNK} chars via blank lines; "
            f"whitespace/newline cleanup only; fixed-{FIXED_SIZE} kept as baseline")

records = []
for p in SOURCES:
    digest = source_hash(p)
    for i, chunk in enumerate(prepared[p.name]["structure"], 1):
        records.append({
            "source_file": p.name,
            "source_sha256": digest,
            "chunk_id": f"{p.stem[:40]}-{i:03d}",
            "chunk_index": i,
            "method": "structure_aware",
            "char_count": len(chunk),
            "text": chunk,
        })

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
with prepared_path.open("w", encoding="utf-8") as out:
    for row in records:
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

# B3 basic output contract
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
assert records and all(r["text"].strip() for r in records), "Empty chunk found"
with prepared_path.open(encoding="utf-8") as fh:
    exported = [json.loads(l) for l in fh if l.strip()]
assert exported == records, "Reloaded export differs from in-memory records"
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size)
print("Validation: file exists, non-empty, reloads identically.")
print("Total chunks exported:", len(records))
print("Representative records:")
for r in (records[1], records[-2]):
    print(" ", r["chunk_id"], "|", repr(r["text"][:120]))

# Repeatability check
rerun = [structure_chunks(clean_text("\n".join(t for _, t in read_pages(p)))) for p in SOURCES]
assert rerun == [prepared[p.name]["structure"] for p in SOURCES]
print("Repeatability: second run identical.")

## B4 – Lineage registration, manifest and verification

In [ ]:
# B4.1 — Register Pathway A (one entry per source -> prepared_text.jsonl)
for p in SOURCES:
    register_lineage(source=p, output=prepared_path, processing=approach)
print("Registered entries:", len(lineage_entries))

In [ ]:
# CELL 6A — Generate common lineage manifest
assert lineage_entries, "No lineage entries registered."
lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source, output, processing = entry["source"], entry["output"], entry["processing"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_dataset": "CUAD v1 (theatticusproject/cuad on Hugging Face)",
        "source_version": "CUAD v1",
        "access_rights": "CC BY 4.0 - attribution: Hendrycks et al. 2021, arXiv:2103.06268",
        "status": "Prepared; quality review pending",
    })
manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))

In [ ]:
# CELL 6B — Verify common lineage manifest
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."
saved = json.loads(manifest_path.read_text(encoding="utf-8"))
saved_records = saved.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."
for rec in saved_records:
    src = RAW_DIR / rec["source_filename"]
    out = BASE_DIR / rec["prepared_output"]
    assert src.is_file(), f"Missing source: {src}"
    assert out.is_file(), f"Missing output: {out}"
    assert source_hash(src) == rec["source_sha256"], f"Source checksum mismatch: {src}"
    assert source_hash(out) == rec["prepared_sha256"], f"Prepared checksum mismatch: {out}"
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)